In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
%pip install mlflow dagshub xgboost

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 91.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 86.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import mlflow
import mlflow.sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import gc
import warnings
warnings.filterwarnings('ignore')

import dagshub
dagshub.init(repo_owner='amama22', 
             repo_name='ieee-cis-fraud-detection', mlflow=True)
mlflow.set_experiment("RandomForest_Training")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=d3267b00-03a3-4423-a3d8-5ce43c962fdb&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=c878161760acdc174dc4e36650c983f769914290a328e511d1220ff4b9f27fe8




Accessing as amama22

Initialized MLflow to track repo "amama22/ieee-cis-fraud-detection"

Repository amama22/ieee-cis-fraud-detection initialized!

<Experiment: artifact_location='mlflow-artifacts:/3da5c5f6d1f54f63ada0baf8e146b80e', creation_time=1778196641630, experiment_id='2', last_update_time=1778196641630, lifecycle_stage='active', name='RandomForest_Training', tags={}, trace_location=None, workspace='default'>

In [4]:
def reduce_memory(df):
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = df[col].astype('int32')
    return df

train = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_id = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_id = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

train = reduce_memory(train.merge(train_id, on='TransactionID', how='left'))
test = reduce_memory(test.merge(test_id, on='TransactionID', how='left'))

TARGET = 'isFraud'
X = train.drop(columns=[TARGET, 'TransactionID'])
y = train[TARGET]
X_test = test.drop(columns=['TransactionID'])

print("Train shape:", X.shape)
print("Memory:", X.memory_usage().sum() / 1e9, "GB")

Train shape: (590540, 432)
Memory: 1.093680212 GB


# cleaning

In [5]:
class RFCleaningTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, null_threshold=0.8):
        self.null_threshold = null_threshold
    
    def fit(self, X, y=None):
        null_pct = X.isnull().sum() / len(X)
        self.high_null_cols_ = null_pct[null_pct > self.null_threshold].index.tolist()
        self.always_drop_ = [c for c in ['M1'] if c in X.columns]
        self.cols_to_drop_ = list(set(self.high_null_cols_ + self.always_drop_))
        
        if 'card6' in X.columns:
            card6_counts = X['card6'].value_counts()
            self.rare_card6_ = card6_counts[card6_counts < 100].index.tolist()
        else:
            self.rare_card6_ = []
        return self
    
    def transform(self, X):
        X = X.copy()
        X = X.drop(columns=[c for c in self.cols_to_drop_ if c in X.columns])
        if 'card6' in X.columns:
            X['card6'] = X['card6'].apply(
                lambda x: 'other' if x in self.rare_card6_ else x
            )
        return X

class RFImputer(BaseEstimator, TransformerMixin):
    def __init__(self, strategy='median', add_missingness_flags=False,
                 flag_threshold=0.20):
        self.strategy = strategy
        self.add_missingness_flags = add_missingness_flags
        self.flag_threshold = flag_threshold
    
    def fit(self, X, y=None):
        self.num_cols_ = X.select_dtypes(include=['number']).columns.tolist()
        self.cat_cols_ = X.select_dtypes(include=['object']).columns.tolist()
        
        self.num_imputer_ = SimpleImputer(strategy=self.strategy)
        if self.num_cols_:
            self.num_imputer_.fit(X[self.num_cols_])
        
        self.cat_imputer_ = SimpleImputer(strategy='most_frequent')
        if self.cat_cols_:
            self.cat_imputer_.fit(X[self.cat_cols_])
        
        if self.add_missingness_flags:
            null_pct = X.isnull().sum() / len(X)
            self.flag_cols_ = null_pct[null_pct > self.flag_threshold].index.tolist()
        else:
            self.flag_cols_ = []
        
        return self
    
    def transform(self, X):
        X = X.copy()
        
        for col in self.flag_cols_:
            if col in X.columns:
                X[f'{col}_was_missing'] = X[col].isnull().astype(int)
        
        if self.num_cols_:
            existing_num = [c for c in self.num_cols_ if c in X.columns]
            X[existing_num] = self.num_imputer_.transform(X[existing_num])
        
        if self.cat_cols_:
            existing_cat = [c for c in self.cat_cols_ if c in X.columns]
            X[existing_cat] = self.cat_imputer_.transform(X[existing_cat])
        
        return X

# feature engineering

In [6]:
class RFFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, version=1):
        self.version = version
    
    def fit(self, X, y=None):
        if self.version >= 2 and 'P_emaildomain' in X.columns:
            self.email_freq_ = X['P_emaildomain'].value_counts(normalize=True).to_dict()
        if self.version == 3 and 'card1' in X.columns:
            self.card1_freq_ = X['card1'].value_counts(normalize=True).to_dict()
            self.global_freq_ = 1 / X['card1'].nunique()
        return self
    
    def transform(self, X):
        X = X.copy()
        
        if 'TransactionAmt' in X.columns:
            X['log_TransactionAmt'] = np.log1p(X['TransactionAmt'])
        
        if 'id_01' in X.columns:
            X['has_identity'] = X['id_01'].isnull().astype(int)
        
        if self.version >= 2:
            if 'P_emaildomain' in X.columns:
                high_risk = ['protonmail.com', 'mail.com', 'outlook.es', 'aim.com']
                mid_risk  = ['outlook.com', 'hotmail.es', 'live.com.mx', 'hotmail.com']
                X['email_risk_tier'] = X['P_emaildomain'].apply(
                    lambda x: 2 if x in high_risk else (1 if x in mid_risk else 0)
                )
            if 'D1' in X.columns:
                X['D1_is_new_account'] = (X['D1'] <= 7).astype(int)
        
        if self.version == 3:
            if 'card1' in X.columns and 'TransactionAmt' in X.columns:
                card1_freq = X['card1'].map(self.card1_freq_).fillna(self.global_freq_)
                X['amt_x_card_freq'] = X['TransactionAmt'] * card1_freq
        
        return X


class RFCategoricalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    
    def fit(self, X, y=None):
        self.m_cols_ = [c for c in X.columns
                        if c.startswith('M') and X[c].dtype == 'object']
        self.other_cat_cols_ = [c for c in X.columns
                                if X[c].dtype == 'object'
                                and c not in self.m_cols_]
        self.ordinal_enc_ = OrdinalEncoder(
            handle_unknown='use_encoded_value',
            unknown_value=-1
        )
        if self.other_cat_cols_:
            self.ordinal_enc_.fit(X[self.other_cat_cols_].astype(str))
        return self
    
    def transform(self, X):
        X = X.copy()
        for col in self.m_cols_:
            if col in X.columns:
                X[col] = X[col].map({'T': 1, 'F': 0}).fillna(-1)
        if self.other_cat_cols_:
            existing = [c for c in self.other_cat_cols_ if c in X.columns]
            X[existing] = self.ordinal_enc_.transform(X[existing].astype(str))
        return X

# feature selection

In [7]:
class VarianceSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.01):
        self.threshold = threshold
    
    def fit(self, X, y=None):
        sel = VarianceThreshold(threshold=self.threshold)
        sel.fit(X)
        self.selected_cols_ = X.columns[sel.get_support()].tolist()
        self.dropped_cols_ = X.columns[~sel.get_support()].tolist()
        return self
    
    def transform(self, X):
        return X[self.selected_cols_]


class CorrelationSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95):
        self.threshold = threshold
    
    def fit(self, X, y=None):
        corr_matrix = X.corr().abs()
        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        self.dropped_cols_ = [col for col in upper.columns
                              if any(upper[col] > self.threshold)]
        self.selected_cols_ = [c for c in X.columns
                               if c not in self.dropped_cols_]
        return self
    
    def transform(self, X):
        return X[self.selected_cols_]


class RFImportanceSelector(BaseEstimator, TransformerMixin):
    def __init__(self, top_n=150):
        self.top_n = top_n
    
    def fit(self, X, y=None):
        sample_size = min(30000, len(X))
        idx = np.random.choice(len(X), size=sample_size, replace=False)
        
        quick_model = RandomForestClassifier(
            n_estimators=50,
            max_depth=8,
            n_jobs=-1,
            random_state=42
        )
        quick_model.fit(X.iloc[idx], y.iloc[idx])
        
        importances = pd.Series(
            quick_model.feature_importances_,
            index=X.columns
        ).sort_values(ascending=False)
        
        self.selected_cols_ = importances.head(self.top_n).index.tolist()
        self.dropped_cols_ = importances.tail(
            len(X.columns) - self.top_n).index.tolist()
        self.importances_ = importances
        return self
    
    def transform(self, X):
        return X[self.selected_cols_]

# training

In [9]:
def process_variation_rf(X, y, version, null_threshold, impute_strategy,
                          add_missingness_flags, selector_class, 
                          selector_params, run_prefix):
    
    with mlflow.start_run(run_name=f"RandomForest_Cleaning_{run_prefix}"):
        cleaner = RFCleaningTransformer(null_threshold=null_threshold)
        cleaner.fit(X)
        X_clean = cleaner.transform(X)
        mlflow.log_param("null_threshold", null_threshold)
        mlflow.log_metric("cols_dropped", len(cleaner.cols_to_drop_))
        mlflow.log_metric("remaining_features", X_clean.shape[1])

    with mlflow.start_run(run_name=f"RandomForest_Feature_Engineering_{run_prefix}"):
        fe = RFFeatureEngineer(version=version)
        fe.fit(X_clean)
        X_fe = fe.transform(X_clean)
        del X_clean; gc.collect()
        mlflow.log_param("version", version)
        mlflow.log_metric("features_after_engineering", X_fe.shape[1])

    with mlflow.start_run(run_name=f"RandomForest_Imputation_{run_prefix}"):
        imputer = RFImputer(
            strategy=impute_strategy,
            add_missingness_flags=add_missingness_flags
        )
        imputer.fit(X_fe)
        X_imp = imputer.transform(X_fe)
        del X_fe; gc.collect()
        mlflow.log_param("strategy", impute_strategy)
        mlflow.log_param("add_missingness_flags", add_missingness_flags)
        mlflow.log_metric("missingness_flags_added", len(imputer.flag_cols_))
        mlflow.log_metric("features_after_imputation", X_imp.shape[1])
        print(f"{run_prefix}: after imputation {X_imp.shape[1]} cols, "
              f"any nulls: {X_imp.isnull().sum().sum()}")

    with mlflow.start_run(run_name=f"RandomForest_Encoding_{run_prefix}"):
        enc = RFCategoricalEncoder()
        enc.fit(X_imp)
        X_enc = enc.transform(X_imp)
        del X_imp; gc.collect()
        mlflow.log_metric("n_cat_cols_encoded", len(enc.other_cat_cols_))

    with mlflow.start_run(run_name=f"RandomForest_Feature_Selection_{run_prefix}"):
        sel = selector_class(**selector_params)
        sel.fit(X_enc, y)
        X_sel = sel.transform(X_enc)
        del X_enc; gc.collect()
        mlflow.log_param("selector", selector_class.__name__)
        mlflow.log_params(selector_params)
        mlflow.log_metric("features_after_selection", X_sel.shape[1])
        mlflow.log_metric("features_dropped", len(sel.dropped_cols_))

    print(f"{run_prefix}: final shape {X_sel.shape}")
    return X_sel, cleaner, fe, imputer, enc, sel

In [10]:
X_sel_v1, cleaner_v1, fe_v1, imp_v1, enc_v1, sel_v1 = process_variation_rf(
    X, y,
    version=1, null_threshold=0.80,
    impute_strategy='median', add_missingness_flags=False,
    selector_class=VarianceSelector, selector_params={"threshold": 0.01},
    run_prefix="V1"
)

X_sel_v2, cleaner_v2, fe_v2, imp_v2, enc_v2, sel_v2 = process_variation_rf(
    X, y,
    version=2, null_threshold=0.50,
    impute_strategy='mean', add_missingness_flags=False,
    selector_class=CorrelationSelector, selector_params={"threshold": 0.95},
    run_prefix="V2"
)

X_sel_v3, cleaner_v3, fe_v3, imp_v3, enc_v3, sel_v3 = process_variation_rf(
    X, y,
    version=3, null_threshold=0.90,
    impute_strategy='median', add_missingness_flags=True,
    selector_class=RFImportanceSelector, selector_params={"top_n": 150},
    run_prefix="V3"
)

🏃 View run RandomForest_Cleaning_V1 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/2/runs/9a206986e3b644979ecee32eb55b1d2f
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/2
🏃 View run RandomForest_Feature_Engineering_V1 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/2/runs/89f9d6cdcb2f45eb9fa3877ee8272c9f
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/2
V1: after imputation 359 cols, any nulls: 0
🏃 View run RandomForest_Imputation_V1 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/2/runs/490449a8089049059de79315bae3cb10
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/2
🏃 View run RandomForest_Encoding_V1 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/2/runs/c127292d136b4b809e63bb59b908bebb
🧪 View experiment at: https://dag

# training

In [13]:
from sklearn.ensemble import RandomForestClassifier

SKF = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

rf_param_sets = {
    "underfit": {
        "n_estimators": 100,
        "max_depth": 5,
        "max_features": "sqrt",
        "min_samples_leaf": 100,
        "class_weight": "balanced",
        "n_jobs": -1,
        "random_state": 42
    },
    "baseline": {
        "n_estimators": 100,
        "max_depth": 15,
        "max_features": "sqrt",
        "min_samples_leaf": 10,
        "class_weight": "balanced",
        "n_jobs": -1,
        "random_state": 42
    },
    "overfit": {
        "n_estimators": 100,
        "max_depth": 30,
        "max_features": 1.0,
        "min_samples_leaf": 1,
        "class_weight": "balanced",
        "n_jobs": -1,
        "random_state": 42
    },
    "tuned": {
        "n_estimators": 200,
        "max_depth": 20,
        "max_features": "sqrt",
        "min_samples_leaf": 20,
        "min_samples_split": 50,
        "class_weight": "balanced",
        "n_jobs": -1,
        "random_state": 42
    }
}

In [12]:
def run_rf_cv(X_data, y_data, params, skf):
    train_aucs, val_aucs = [], []
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_data, y_data)):
        X_tr = X_data.iloc[train_idx]
        X_val = X_data.iloc[val_idx]
        y_tr = y_data.iloc[train_idx]
        y_val = y_data.iloc[val_idx]
        
        model = RandomForestClassifier(**params)
        model.fit(X_tr, y_tr)
        
        train_aucs.append(roc_auc_score(y_tr, model.predict_proba(X_tr)[:,1]))
        val_aucs.append(roc_auc_score(y_val, model.predict_proba(X_val)[:,1]))
        
        del X_tr, X_val, model; gc.collect()
    
    return np.mean(train_aucs), np.mean(val_aucs)

In [ ]:
datasets = {"V1": X_sel_v1, "V2": X_sel_v2, "V3": X_sel_v3}

for var_name, X_data in datasets.items():
    for param_name, params in rf_param_sets.items():
        run_name = f"RandomForest_CV_{var_name}_{param_name}"
        print(f"Running {run_name}...")
        
        with mlflow.start_run(run_name=run_name):
            train_auc, val_auc = run_rf_cv(X_data, y, params, SKF)
            gap = train_auc - val_auc
            
            mlflow.log_params(params)
            mlflow.log_param("variation", var_name)
            mlflow.log_param("n_features", X_data.shape[1])
            mlflow.log_metric("train_auc", train_auc)
            mlflow.log_metric("val_auc", val_auc)
            mlflow.log_metric("overfit_gap", gap)
            
            print(f"  train={train_auc:.4f} | val={val_auc:.4f} | gap={gap:.4f}")

Running RandomForest_CV_V1_underfit...
  train=0.8524 | val=0.8507 | gap=0.0017
🏃 View run RandomForest_CV_V1_underfit at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/2/runs/91072d4b265a4926832200c738d2379d
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/2
Running RandomForest_CV_V1_baseline...
  train=0.9482 | val=0.9114 | gap=0.0368
🏃 View run RandomForest_CV_V1_baseline at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/2/runs/f44ec77462ee4367ba4829934b021ddc
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/2
Running RandomForest_CV_V1_overfit...


In [8]:
rf_best_params = {
    "n_estimators": 200,
    "max_depth": 20,
    "max_features": "sqrt",
    "min_samples_leaf": 20,
    "min_samples_split": 50,
    "class_weight": "balanced",
    "n_jobs": -1,
    "random_state": 42
}

rf_best_pipeline = Pipeline([
    ('cleaner', RFCleaningTransformer(null_threshold=0.50)),
    ('feature_engineer', RFFeatureEngineer(version=2)),
    ('imputer', RFImputer(strategy='mean', add_missingness_flags=False)),
    ('encoder', RFCategoricalEncoder()),
    ('selector', CorrelationSelector(threshold=0.95)),
    ('model', RandomForestClassifier(**rf_best_params))
])

rf_best_pipeline.fit(X, y)

with mlflow.start_run(run_name="RandomForest_Best_Pipeline"):
    mlflow.log_params(rf_best_params)
    mlflow.log_param("null_threshold", 0.50)
    mlflow.log_param("impute_strategy", "mean")
    mlflow.log_param("fe_version", 2)
    mlflow.log_param("selector", "CorrelationFilter_0.95")
    mlflow.log_param("variation", "V2")
    mlflow.log_metric("val_auc", 0.9249)
    mlflow.log_metric("overfit_gap", 0.0300)
    mlflow.sklearn.log_model(
        sk_model=rf_best_pipeline,
        artifact_path="rf_fraud_pipeline"
    )
    print("RandomForest pipeline saved to MLflow.")

2026/05/08 12:33:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/08 12:34:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RandomForest pipeline saved to MLflow.
🏃 View run RandomForest_Best_Pipeline at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/2/runs/afa6f35d535847a5908d877c5fea491d
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/2
